# Step 7 - Airline Delay Feature Experiment

This notebook tests a small new feature family before changing the main pipeline.

Goal:
- keep the current weather-aware Random Forest setup
- add rolling airline delay features
- check whether they improve the 2022 train / 2023 test result


## What This Uses

No new raw data is needed for this step.

The rolling airline features come from the existing flight logs using only past flights in time order.

In [8]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

## Load And Clean Flight Data


In [9]:
flight_dir = Path("../data/raw/flight")
csv_files = sorted(flight_dir.glob("**/*_flight.csv"))

flight_cols = [
    "Month",
    "FlightDate",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepDelay",
    "Cancelled"
]

flight_parts = [pd.read_csv(file, usecols=flight_cols) for file in csv_files]
flights = pd.concat(flight_parts, ignore_index=True)
flights.shape

(13577024, 9)

In [10]:
flights = flights[flights["Cancelled"] == 0].copy()
flights = flights.dropna(subset=["FlightDate", "CRSDepTime", "DepDelay", "Reporting_Airline", "Origin", "Dest"])

flights["Delay"] = (flights["DepDelay"] > 15).astype(int)
flights = flights.drop(columns=["DepDelay"])

flights["FlightDate"] = pd.to_datetime(flights["FlightDate"])
flights["CRSDepTime"] = flights["CRSDepTime"].astype(int)
flights["dep_hour"] = flights["CRSDepTime"] // 100

dep_minutes = (flights["CRSDepTime"] // 100) * 60 + (flights["CRSDepTime"] % 100)
flights["scheduled_departure_local"] = flights["FlightDate"] + pd.to_timedelta(dep_minutes, unit="m")

flights["route"] = flights["Origin"] + "_" + flights["Dest"]
flights["is_weekend"] = flights["DayOfWeek"].isin([6, 7]).astype(int)
flights["time_of_day_bin"] = pd.cut(
    flights["dep_hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["overnight", "morning", "afternoon", "evening", "night"]
)

flights.shape

(13307795, 14)

## Load Weather And Build The Current Best Setup


In [11]:
weather_paths = [
    Path("../data/raw/weather/asos_top20_origins_2022.csv"),
    Path("../data/raw/weather/asos_top20_origins_2023.csv"),
]

weather_parts = []
for path in weather_paths:
    weather_part = pd.read_csv(path)
    weather_part["valid"] = pd.to_datetime(weather_part["valid"])
    weather_parts.append(weather_part)

weather = pd.concat(weather_parts, ignore_index=True)
weather.shape

(408714, 15)

In [12]:
weather_stations = sorted(weather["station"].dropna().unique().tolist())
flights = flights[flights["Origin"].isin(weather_stations)].copy()

AIRPORT_TIMEZONES = {
    "ATL": "America/New_York",
    "BOS": "America/New_York",
    "CLT": "America/New_York",
    "DCA": "America/New_York",
    "DTW": "America/New_York",
    "EWR": "America/New_York",
    "JFK": "America/New_York",
    "LGA": "America/New_York",
    "MCO": "America/New_York",
    "MIA": "America/New_York",
    "DFW": "America/Chicago",
    "IAH": "America/Chicago",
    "MSP": "America/Chicago",
    "ORD": "America/Chicago",
    "DEN": "America/Denver",
    "PHX": "America/Phoenix",
    "LAS": "America/Los_Angeles",
    "LAX": "America/Los_Angeles",
    "SEA": "America/Los_Angeles",
    "SFO": "America/Los_Angeles",
}

flights.shape

(6924161, 14)

In [13]:
def add_scheduled_departure_utc(flight_df, timezone_map):
    pieces = []
    for airport, group in flight_df.groupby("Origin", group_keys=False):
        tz_name = timezone_map[airport]
        group = group.copy()
        localized = group["scheduled_departure_local"].dt.tz_localize(
            tz_name,
            nonexistent="shift_forward",
            ambiguous="NaT"
        )
        group["scheduled_departure_utc"] = localized.dt.tz_convert("UTC").dt.tz_localize(None)
        pieces.append(group)
    return pd.concat(pieces, ignore_index=True)


flights = add_scheduled_departure_utc(flights, AIRPORT_TIMEZONES)
flights = flights.dropna(subset=["scheduled_departure_utc"]).copy()
flights[["Origin", "scheduled_departure_local", "scheduled_departure_utc"]].head()


,Origin,scheduled_departure_local,scheduled_departure_utc
0,ATL,2022-01-04 14:39:00,2022-01-04 19:39:00
1,ATL,2022-01-05 14:39:00,2022-01-05 19:39:00
2,ATL,2022-01-06 14:39:00,2022-01-06 19:39:00
3,ATL,2022-01-07 14:39:00,2022-01-07 19:39:00
4,ATL,2022-01-08 14:00:00,2022-01-08 19:00:00


In [14]:
def join_weather_to_flights(flight_df, weather_df, tolerance_hours=2):
    joined_parts = []
    tolerance = pd.Timedelta(hours=tolerance_hours)

    for station in sorted(set(flight_df["Origin"]).intersection(set(weather_df["station"]))):
        flight_part = flight_df[flight_df["Origin"] == station].sort_values("scheduled_departure_utc").copy()
        weather_part = weather_df[weather_df["station"] == station].sort_values("valid").copy()

        merged = pd.merge_asof(
            flight_part,
            weather_part,
            left_on="scheduled_departure_utc",
            right_on="valid",
            direction="backward",
            tolerance=tolerance
        )
        joined_parts.append(merged)

    joined = pd.concat(joined_parts, ignore_index=True)
    joined["weather_report_age_minutes"] = (
        joined["scheduled_departure_utc"] - joined["valid"]
    ).dt.total_seconds().div(60)
    return joined


model_df = join_weather_to_flights(flights, weather, tolerance_hours=2)
model_df.shape

(6924147, 31)

In [15]:
model_df["gust"] = model_df["gust"].fillna(0)
model_df["p01i"] = model_df["p01i"].fillna(0)
model_df["vsby"] = model_df["vsby"].fillna(10)
model_df["wxcodes"] = model_df["wxcodes"].fillna("")
model_df["skyc1"] = model_df["skyc1"].fillna("CLR")

model_df["has_precip"] = (model_df["p01i"] > 0).astype(int)
model_df["low_visibility"] = (model_df["vsby"] < 3).astype(int)
model_df["high_wind"] = (model_df["sknt"].fillna(0) >= 15).astype(int)
model_df["has_gust"] = (model_df["gust"] > 0).astype(int)
model_df["has_weather_code"] = (model_df["wxcodes"] != "").astype(int)

required_weather_cols = ["valid", "tmpf", "relh", "sknt", "alti", "weather_report_age_minutes"]
model_df = model_df.dropna(subset=required_weather_cols).copy()
model_df.shape

(6917948, 36)

## Add Rolling Airline Delay Features

These only use past flights in time order. They do not use the current flight or future flights.

In [16]:
model_df = model_df.sort_values("scheduled_departure_utc").copy()

def add_rolling_delay_features(df, group_cols, prefix, window="3h"):
    parts = []
    for _, group in df.groupby(group_cols, group_keys=False):
        group = group.sort_values("scheduled_departure_utc").copy()
        ts = group.set_index("scheduled_departure_utc")
        delay_shifted = ts["Delay"].shift(1)

        ts[f"{prefix}_delay_rate_prev_3h"] = delay_shifted.rolling(window, min_periods=1).mean()
        ts[f"{prefix}_delay_count_prev_3h"] = delay_shifted.rolling(window, min_periods=1).sum()
        ts[f"{prefix}_flight_count_prev_3h"] = delay_shifted.rolling(window, min_periods=1).count()

        ts = ts.reset_index()
        parts.append(ts)

    return pd.concat(parts, ignore_index=True)


model_df = add_rolling_delay_features(model_df, ["Reporting_Airline"], "airline")
model_df = add_rolling_delay_features(model_df, ["Reporting_Airline", "Origin"], "airline_origin")

rolling_feature_cols = [
    "airline_delay_rate_prev_3h",
    "airline_delay_count_prev_3h",
    "airline_flight_count_prev_3h",
    "airline_origin_delay_rate_prev_3h",
    "airline_origin_delay_count_prev_3h",
    "airline_origin_flight_count_prev_3h",
]

for col in rolling_feature_cols:
    model_df[col] = model_df[col].fillna(0)

model_df[rolling_feature_cols].head()

,airline_delay_rate_prev_3h,airline_delay_count_prev_3h,airline_flight_count_prev_3h,airline_origin_delay_rate_prev_3h,airline_origin_delay_count_prev_3h,airline_origin_flight_count_prev_3h
0,0.250000,2.0,8.0,0.000000,0.0,0.0
1,0.222222,2.0,9.0,0.000000,0.0,1.0
2,0.166667,2.0,12.0,0.000000,0.0,2.0
3,0.150000,3.0,20.0,0.333333,1.0,3.0
4,0.125000,3.0,24.0,0.250000,1.0,4.0


## Train/Test Split

Train on 2022. Test on 2023.

In [17]:
train_df = model_df[model_df["FlightDate"].dt.year == 2022].copy()
test_df = model_df[model_df["FlightDate"].dt.year == 2023].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train delay rate:", train_df["Delay"].mean())
print("Test delay rate:", test_df["Delay"].mean())

Train shape: (3398442, 42)
Test shape: (3519506, 42)
Train delay rate: 0.21390507767971323
Test delay rate: 0.21067615739254317


In [18]:
def add_delay_rate_feature(train_data, test_data, group_col, target_col, new_col):
    global_rate = train_data[target_col].mean()
    rates = train_data.groupby(group_col)[target_col].mean()
    train_data = train_data.copy()
    test_data = test_data.copy()
    train_data[new_col] = train_data[group_col].map(rates).fillna(global_rate)
    test_data[new_col] = test_data[group_col].map(rates).fillna(global_rate)
    return train_data, test_data


train_df, test_df = add_delay_rate_feature(train_df, test_df, "Origin", "Delay", "origin_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "Reporting_Airline", "Delay", "airline_delay_rate")
train_df, test_df = add_delay_rate_feature(train_df, test_df, "route", "Delay", "route_delay_rate")

## Random Forest Comparison

This compares the current weather-aware feature set against the same setup with rolling airline features added.

In [19]:
base_numeric_features = [
    "DayOfWeek",
    "dep_hour",
    "is_weekend",
    "origin_delay_rate",
    "airline_delay_rate",
    "route_delay_rate",
    "tmpf",
    "relh",
    "sknt",
    "alti",
    "vsby",
    "weather_report_age_minutes",
    "p01i",
]

base_categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "route",
    "time_of_day_bin",
]

airline_rolling_features = rolling_feature_cols

feature_sets = {
    "Current Weather Model": {
        "numeric": base_numeric_features,
        "categorical": base_categorical_features,
    },
    "Weather + Airline Rolling Features": {
        "numeric": base_numeric_features + airline_rolling_features,
        "categorical": base_categorical_features,
    },
}

In [20]:
def build_random_forest_pipeline(numeric_features, categorical_features):
    preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="most_frequent"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=150,
            max_depth=14,
            min_samples_leaf=5,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])
    return model


results = []

for name, spec in feature_sets.items():
    features = spec["numeric"] + spec["categorical"]
    X_train = train_df[features]
    y_train = train_df["Delay"]
    X_test = test_df[features]
    y_test = test_df["Delay"]

    model = build_random_forest_pipeline(spec["numeric"], spec["categorical"])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "experiment": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_delay": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_delay": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_delay": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    })

    print("=" * 80)
    print(name)
    print(classification_report(y_test, y_pred, zero_division=0))
    print(confusion_matrix(y_test, y_pred))

pd.DataFrame(results).sort_values("f1_delay", ascending=False)

Current Weather Model
              precision    recall  f1-score   support

           0       0.86      0.65      0.74   2778030
           1       0.31      0.59      0.41    741476

    accuracy                           0.64   3519506
   macro avg       0.59      0.62      0.58   3519506
weighted avg       0.74      0.64      0.67   3519506

[[1818969  959061]
 [ 301091  440385]]
Weather + Airline Rolling Features
              precision    recall  f1-score   support

           0       0.88      0.72      0.79   2778030
           1       0.37      0.62      0.47    741476

    accuracy                           0.70   3519506
   macro avg       0.62      0.67      0.63   3519506
weighted avg       0.77      0.70      0.72   3519506

[[1999550  778480]
 [ 280371  461105]]


,experiment,accuracy,precision_delay,recall_delay,f1_delay
1,Weather + Airline Rolling Features,0.699148,0.371983,0.621874,0.465513
0,Current Weather Model,0.641952,0.314685,0.593930,0.411398
